# Segmentasi Citra Biomedis
**Proyek Akhir – Segmentasi Citra Biomedis**

Notebook ini membandingkan dua metode segmentasi:
1. **Otsu Thresholding** + Morphological Processing (dengan deteksi border hitam)
2. **K-Means Clustering** pada ruang warna Lab (K=3 dan K=4)

Evaluasi dilakukan dengan metrik IoU, Dice, Precision, dan Recall
terhadap ground truth mask.

In [14]:
# ==========================================
# 1. IMPORT LIBRARY & SETUP (LOCAL)
# ==========================================
import os
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')          # backend non-GUI agar aman saat loop banyak gambar
import matplotlib.pyplot as plt

In [15]:
BASE_DIR      = r"C:\De\Dokumen\Project Akhir Citra\analisis-citra-polip-endoskopi\Berhasil\Datasets"
FOLDER_IMAGES = os.path.join(BASE_DIR, "Images")
FOLDER_MASKS  = os.path.join(BASE_DIR, "Masks")

OUTPUT_BASE = r"C:\De\Dokumen\Project Akhir Citra\analisis-citra-polip-endoskopi\Berhasil"
OUTPUT_P1   = os.path.join(OUTPUT_BASE, "Percobaan_1")
OUTPUT_P2   = os.path.join(OUTPUT_BASE, "Percobaan_2")
OUTPUT_P3   = os.path.join(OUTPUT_BASE, "Percobaan_3")

# cukup folder percobaan saja — tanpa subfolder
for folder in [OUTPUT_P1, OUTPUT_P2, OUTPUT_P3]:
    os.makedirs(folder, exist_ok=True)

assert os.path.isdir(FOLDER_IMAGES), f"Folder tidak ditemukan: {FOLDER_IMAGES}"
assert os.path.isdir(FOLDER_MASKS),  f"Folder tidak ditemukan: {FOLDER_MASKS}"

files = sorted(os.listdir(FOLDER_IMAGES))
print("Jumlah gambar :", len(files))

gambar = {}

rata_akurasi = {}

gambar2 = {}

rata_akurasi2 = {}

Jumlah gambar : 25


In [16]:
# ==========================================
# 3. UTILITY FUNCTIONS
# ==========================================
def load_image(filename):
    """Load image (RGB) dan ground-truth mask (grayscale)."""
    path_img  = os.path.join(FOLDER_IMAGES, filename)
    path_gt   = os.path.join(FOLDER_MASKS, filename)

    img_bgr = cv2.imread(path_img)
    if img_bgr is None:
        raise FileNotFoundError(f"Gambar tidak ditemukan: {path_img}")

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    mask_gt = cv2.imread(path_gt, cv2.IMREAD_GRAYSCALE)
    return img_bgr, img_rgb, gray, mask_gt


def detect_black_border(gray, thresh_dark=15):
    """Deteksi border hitam di tepi citra → return valid_mask (bool)."""
    dark = (gray <= thresh_dark).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(dark, 8)

    h, w = gray.shape
    border_mask = np.zeros_like(gray)
    for i in range(1, n):
        x, y, bw, bh, _ = stats[i]
        if x == 0 or y == 0 or x + bw >= w or y + bh >= h:
            border_mask[labels == i] = 255

    return border_mask == 0  # True = area valid


def otsu_on_valid_area(gray, valid_mask):
    """Otsu threshold hanya dihitung pada piksel area valid."""
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    valid_px = blurred[valid_mask].reshape(-1, 1)

    thr_val, _ = cv2.threshold(valid_px, 0, 255,
                               cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    thresh = np.zeros_like(gray)
    thresh[(blurred <= thr_val) & valid_mask] = 255
    return blurred, thresh, thr_val


def evaluate_mask(pred, gt):
    """Hitung metrik segmentasi. Input: binary mask (0/255)."""
    if gt is None:
        return None
    p = (pred > 127).astype(np.uint8)
    g = (gt   > 127).astype(np.uint8)

    inter = np.logical_and(p, g).sum()
    union = np.logical_or(p, g).sum()

    iou  = inter / union if union > 0 else 0.0
    dice = 2 * inter / (p.sum() + g.sum()) if (p.sum() + g.sum()) > 0 else 0.0
    prec = inter / p.sum() if p.sum() > 0 else 0.0
    rec  = inter / g.sum() if g.sum() > 0 else 0.0

    return {"IoU": iou, "Dice": dice, "Precision": prec, "Recall": rec}

## Percobaan 1 — Otsu Thresholding + Morphology

**Alur:**
1. Deteksi border hitam → area valid
2. Gaussian Blur (5×5)
3. Otsu threshold **hanya** pada area valid
4. Morphological Opening (hapus noise kecil) + Closing (tutup lubang)
5. Balik polaritas agar objek terang menjadi foreground

In [31]:
# ==========================================
# PERCOBAAN 1 — OTSU THRESHOLDING (SEMUA GAMBAR)
# ==========================================
kernel = np.ones((5, 5), np.uint8)
files = sorted(os.listdir(FOLDER_IMAGES))
all_metrics_p1 = []

threshold_akurasi = 0.7

jumlah_lolos = {"Otsu" : 0, "Otsu_Morph" : 0, "Total" : 0}

akurasi_otsu = []
akurasi_otsu_morph = []
nama_gambar = []

for nama_file in files:
    stem = os.path.splitext(nama_file)[0]
    
    # --- load ---
    img_bgr, img_rgb, gray, mask_gt = load_image(nama_file)

    # --- proses ---
    valid_mask = detect_black_border(gray)
    blurred, thresh, thr_val = otsu_on_valid_area(gray, valid_mask)
    mask_inverted = np.where(valid_mask == 1, 255 - thresh, thresh)
    cv2.imwrite(os.path.join(OUTPUT_P1, "overview", f"{stem}_Otsu.png"),mask_inverted)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel)
    closing = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    segmentation_mask = cv2.bitwise_not(closing)
    segmentation_mask[~valid_mask] = 0
    cv2.imwrite(os.path.join(OUTPUT_P1, "overview", f"{stem}_Otsu_Morph.png"),segmentation_mask)
    segmented = cv2.bitwise_and(img_rgb, img_rgb, mask=segmentation_mask)

    # --- evaluasi ---
    m = evaluate_mask(segmentation_mask, mask_gt)
    all_metrics_p1.append(m)

    # 1) mask segmentasi (B/W) — pakai ini untuk IoU/dice ulang nanti
    cv2.imwrite(os.path.join(OUTPUT_P1, "mask", f"{stem}_mask.png"), segmentation_mask)

    # 2) gambar objek tersegmentasi (RGB di atas hitam)
    cv2.imwrite(os.path.join(OUTPUT_P1, "segmented", f"{stem}_seg.png"),
                cv2.cvtColor(segmented, cv2.COLOR_RGB2BGR))

    # 3) overview 6-panel (seperti di Colab)
    fig, axes = plt.subplots(2, 4, figsize=(18, 10))
    panels = [
        (img_rgb,           "Original Image",                None),
        (valid_mask,        "Valid Area",                    "gray"),
        (blurred,           "Gaussian Blur",                 "gray"),
        (segmented,         "Segmented Object",              None),
        (mask_inverted,     "Otsu Segmentation",             "gray"),
        (segmentation_mask, "MorphologyEx Noise Reduction",  "gray"),
        (mask_gt,           "Ground Truth",                  "gray"),
    ]
    
    fig.suptitle(f"Gambar {stem}",size = "xx-large", weight = "bold", family = "Times New Roman")

    for ax, (im, title, cmap) in zip(axes.flat, panels):
        ax.imshow(im, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")

    # Sembunyikan slot ke-8 (index ke-7) karena cuma ada 7 panel
    for ax in axes.flat[len(panels):]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(os.path.join(OUTPUT_BASE, "Percobaan_1_Final"), f"{stem}_final.png"),
                dpi=100, bbox_inches="tight")
    plt.close(fig)   # WAJIB agar tidak memory leak saat loop ratusan gambar
    
    #----------------------------------------------------------------------------------
    
    if np.mean(mask_inverted==mask_gt) > threshold_akurasi :
        jumlah_lolos["Otsu"] += 1
    if np.mean(segmentation_mask==mask_gt) > threshold_akurasi :
        jumlah_lolos["Otsu_Morph"] += 1
    jumlah_lolos["Total"] += 1
    akurasi_otsu.append(np.mean(mask_inverted==mask_gt))
    akurasi_otsu_morph.append(np.mean(segmentation_mask==mask_gt))
    
    nama_gambar.append(stem)
    
    #----------------------------------------------------------------------------------

# --- ringkasan rata-rata ---
keys = ["IoU", "Dice", "Precision", "Recall"]
print("\n=== Rata-rata Metrik Percobaan 1 (Otsu) ===")
for k in keys:
    vals = [m[k] for m in all_metrics_p1 if m]
    print(f"  {k:10s}: {np.mean(vals):.4f}")
print(f"Akurasi Berdasarkan Jumlah Gambar (threshold = {threshold_akurasi}) :\nOtsu\t\t: {jumlah_lolos['Otsu']/jumlah_lolos['Total']}\nOtsu_Morph\t: {jumlah_lolos['Otsu_Morph']/jumlah_lolos['Total']}")
print(f"Rata2 Akurasi :\nOtsu\t\t: {np.mean(akurasi_otsu)}\nOtsu_Morph\t: {np.mean(akurasi_otsu_morph)}")

gambar["Otsu (Gray)"] = jumlah_lolos['Otsu']/jumlah_lolos['Total']
gambar["Otsu_Morph (Gray)"] = jumlah_lolos['Otsu_Morph']/jumlah_lolos['Total']

rata_akurasi["Otsu (Gray)"] = np.mean(akurasi_otsu)
rata_akurasi["Otsu_Morph (Gray)"] = np.mean(akurasi_otsu_morph)

akurasi_otsu_p1 = akurasi_otsu
akurasi_otsu_morph_p1 = akurasi_otsu_morph




=== Rata-rata Metrik Percobaan 1 (Otsu) ===
  IoU       : 0.5517
  Dice      : 0.6958
  Precision : 0.6579
  Recall    : 0.8109
Akurasi Berdasarkan Jumlah Gambar (threshold = 0.7) :
Otsu		: 0.96
Otsu_Morph	: 0.96
Rata2 Akurasi :
Otsu		: 0.8365955530730731
Otsu_Morph	: 0.8415782966846703


## Percobaan 2 — K-Means Clustering (Lab)

Menggunakan **ruang warna Lab** (L, a, b) agar pemisahan foreground–background
lebih perseptual. Diuji dengan **K = 3** dan **K = 4**.

In [18]:
# ==========================================
# PERCOBAAN 2 — K-MEANS CLUSTERING (SEMUA GAMBAR)
# ==========================================
all_metrics_p2 = {3: [], 4: []}

jumlah_lolos = {"K3 C1" : 0, "K3 C2" : 0,"K4 C1" : 0,"K4 C2" : 0, "Total" : 0}

akurasi_cluster = {"K3 C1" : [], "K3 C2" : [],"K4 C1" : [],"K4 C2" : []}

for nama_file in files:
    img_bgr, img_rgb, gray, mask_gt = load_image(nama_file)
    valid_mask = detect_black_border(gray)

    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    pixels = lab[valid_mask].reshape(-1, 3).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)

    hasil_kmeans = []
    for K in [3, 4]:
        compactness, labels, centers = cv2.kmeans(
            pixels, K, None, criteria, 10, cv2.KMEANS_PP_CENTERS
        )
        cluster_map = np.zeros(gray.shape, dtype=np.uint8)
        cluster_map[valid_mask] = labels.flatten()
        hasil_kmeans.append((K, cluster_map))

    stem = os.path.splitext(nama_file)[0]

    # --- Visualisasi 3x3 (seperti di Colab) & simpan ---
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    axes[0, 0].imshow(img_rgb);                   axes[0, 0].set_title("Original Image")
    axes[0, 1].imshow(lab[:, :, 1], cmap="gray"); axes[0, 1].set_title("Lab – Channel A")
    axes[0, 2].imshow(lab[:, :, 2], cmap="gray"); axes[0, 2].set_title("Lab – Channel B")

    for row, (K, cmap_arr) in enumerate(hasil_kmeans, start=1):
        axes[row, 0].imshow(cmap_arr, cmap="tab10", vmin=0, vmax=K - 1)
        axes[row, 0].set_title(f"K-Means K={K}")
        for cid in range(1, min(K, 3)):
            selected = (cmap_arr == cid)
            result = np.zeros_like(img_rgb)
            result[selected] = img_rgb[selected]
            axes[row, cid].imshow(result)
            mask_inverted = np.where(result == 0, 0, 255)
            if len(mask_inverted.shape) == 3:
                mask_inverted = mask_inverted[:, :, 0]
            else:
                mask_inverted = mask_inverted
            if np.mean(mask_inverted==mask_gt) > threshold_akurasi :
                jumlah_lolos[f"K{K} C{cid}"] += 1
            akurasi_cluster[f"K{K} C{cid}"].append(np.mean(mask_inverted==mask_gt)) 
            cv2.imwrite(os.path.join(OUTPUT_P2, "overview", f"{stem} K={K} Cluster {cid}.png"),mask_inverted)
            axes[row, cid].set_title(f"K={K}, Cluster {cid}")
    jumlah_lolos["Total"] += 1
    for ax in axes.flat: ax.axis("off")
    plt.tight_layout()
    # plt.savefig(os.path.join(OUTPUT_P2, "overview", f"{stem}_kmeans.png"),
    #             dpi=100, bbox_inches="tight")
    plt.close(fig)

    # --- Simpan tiap cluster K=3 & K=4 sbg mask biner terpisah ---
    for K, cmap_arr in hasil_kmeans:
        for cid in range(K):
            binary = ((cmap_arr == cid) & valid_mask).astype(np.uint8) * 255
            cv2.imwrite(
                os.path.join(OUTPUT_P2, "mask", f"{stem}_K{K}_cluster{cid}.png"),
                binary
            )
            # evaluasi setiap cluster — nanti pilih yang terbaik (IoU tertinggi)
            m = evaluate_mask(binary, mask_gt)
            all_metrics_p2[K].append(m)

print("\n=== Rata-rata Metrik Percobaan 2 (K-Means) ===")
for K in [3, 4]:
    for k in keys:
        vals = [m[k] for m in all_metrics_p2[K]]
        print(f"  K={K} {k:10s}: {np.mean(vals):.4f}")
        
print(f"Akurasi Berdasarkan Jumlah Gambar (threshold = {threshold_akurasi}) :")
for i in range(3,5) :
    for j in range(1,3) :
        print(f"K{i} C{j}\t: {jumlah_lolos[f'K{i} C{j}']/jumlah_lolos['Total']}")
        gambar2[f"K{i} C{j}"] = jumlah_lolos[f'K{i} C{j}']/jumlah_lolos['Total']
        
print(f"Rata2 Akurasi :")
for i in range(3,5) :
    for j in range(1,3) :
        print(f"K{i} C{j}\t: {np.mean(akurasi_cluster[f'K{i} C{j}'])}")
        rata_akurasi2[f"K{i} C{j}"] = np.mean(akurasi_cluster[f'K{i} C{j}'])


=== Rata-rata Metrik Percobaan 2 (K-Means) ===
  K=3 IoU       : 0.2271
  K=3 Dice      : 0.3128
  K=3 Precision : 0.3314
  K=3 Recall    : 0.3332
  K=4 IoU       : 0.1869
  K=4 Dice      : 0.2706
  K=4 Precision : 0.3490
  K=4 Recall    : 0.2499
Akurasi Berdasarkan Jumlah Gambar (threshold = 0.7) :
K3 C1	: 0.36
K3 C2	: 0.36
K4 C1	: 0.4
K4 C2	: 0.52
Rata2 Akurasi :
K3 C1	: 0.6101475987904563
K3 C2	: 0.6431873498715822
K4 C1	: 0.6503322224392662
K4 C2	: 0.7165279918942292


## Percobaan 3 — Segmentasi pada Channel Cr (YCrCb)

Channel **Cr** merepresentasikan chrominance merah dan cukup stabil
terhadap variasi pencahayaan. Dari visualisasi awal, objek tampak lebih
gelap dibanding background pada channel ini, sehingga kita gunakan
`THRESH_BINARY_INV` (objek = nilai Cr kecil).

**Sub-percobaan:**
- **Opsi 1**: Otsu thresholding langsung pada channel Cr.
- **Opsi 2**: Otsu + Morphological Opening (hapus noise) + Closing (tutup lubang).

In [19]:
# ==========================================
# PERCOBAAN 3 — SEGMENTASI PADA CHANNEL Cr (YCrCb)
# LOOP SEMUA GAMBAR + SIMPAN HASIL
# ==========================================
all_metrics_p3 = {
    "Opsi1_Otsu":   [],
    "Opsi2_Morph":  [],
}

kernel = np.ones((7, 7), np.uint8)

jumlah_lolos = {"Otsu" : 0, "Otsu_Morph" : 0, "Total" : 0}

akurasi_otsu = []
akurasi_otsu_morph = []

for nama_file in files:
    # --- load ---
    img_bgr, img_rgb, gray, mask_gt = load_image(nama_file)

    # 1. Border detection
    valid_mask = detect_black_border(gray)

    # 2. Konversi ke YCrCb & ambil channel Cr
    ycrcb      = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
    cr_channel = ycrcb[:, :, 1]

    # 3. Gaussian blur ringan
    cr_blur = cv2.GaussianBlur(cr_channel, (5, 5), 0)

    # ==========================================
    # OPSI 1 — Otsu langsung pada channel Cr
    # ==========================================
    valid_px = cr_blur[valid_mask].reshape(-1, 1)
    thr_val, _ = cv2.threshold(
        valid_px, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    mask_otsu_cr = np.zeros_like(cr_channel)
    mask_otsu_cr[(cr_blur >= thr_val) & valid_mask] = 255

    # ==========================================
    # OPSI 2 — Otsu + Morphology
    # ==========================================
    closing = cv2.morphologyEx(mask_otsu_cr, cv2.MORPH_CLOSE, kernel)
    opening = cv2.morphologyEx(closing,       cv2.MORPH_OPEN,  kernel)

    mask_clean_cr = opening.copy()
    mask_clean_cr[~valid_mask] = 0

    # ==========================================
    # HASIL TERSEMENGTASI
    # ==========================================
    segmented_otsu  = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_otsu_cr)
    segmented_clean = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_clean_cr)

    # ==========================================
    # EVALUASI
    # ==========================================
    met_otsu  = evaluate_mask(mask_otsu_cr,  mask_gt)
    met_clean = evaluate_mask(mask_clean_cr, mask_gt)
    all_metrics_p3["Opsi1_Otsu"].append(met_otsu)
    all_metrics_p3["Opsi2_Morph"].append(met_clean)

    # ==========================================
    # SIMPAN HASIL KE DISK
    # ==========================================
    stem = os.path.splitext(nama_file)[0]

    # mask biner
    cv2.imwrite(os.path.join(OUTPUT_P3, "mask", f"{stem}_opsi1_otsu.png"),  mask_otsu_cr)
    cv2.imwrite(os.path.join(OUTPUT_P3, "mask", f"{stem}_opsi2_morph.png"), mask_clean_cr)

    # gambar tersegmentasi
    cv2.imwrite(os.path.join(OUTPUT_P3, "segmented", f"{stem}_opsi1_seg.png"),
                cv2.cvtColor(segmented_otsu,  cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(OUTPUT_P3, "segmented", f"{stem}_opsi2_seg.png"),
                cv2.cvtColor(segmented_clean, cv2.COLOR_RGB2BGR))

    # overview 2x5 (seperti di Colab)
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))

    axes[0, 0].imshow(img_rgb);                 axes[0, 0].set_title("Original")
    axes[0, 1].imshow(cr_blur, cmap="gray");    axes[0, 1].set_title("Cr Channel (blur)")
    axes[0, 2].imshow(mask_otsu_cr, cmap="gray"); axes[0, 2].set_title(f"Opsi 1 – Otsu\n(thr={thr_val:.1f})")
    axes[0, 3].imshow(segmented_otsu);          axes[0, 3].set_title("Opsi 1 – Segmented")
    axes[0, 4].imshow(mask_gt, cmap="gray");    axes[0, 4].set_title("Ground Truth")

    axes[1, 0].imshow(img_rgb);                 axes[1, 0].set_title("Original")
    axes[1, 1].imshow(opening, cmap="gray");    axes[1, 1].set_title("Opening")
    axes[1, 2].imshow(mask_clean_cr, cmap="gray"); axes[1, 2].set_title("Opsi 2 – Otsu + Morph")
    axes[1, 3].imshow(segmented_clean);         axes[1, 3].set_title("Opsi 2 – Segmented")
    axes[1, 4].imshow(mask_gt, cmap="gray");    axes[1, 4].set_title("Ground Truth")
    
    #----------------------------------------------------------------------------------
        
    if np.mean(mask_otsu_cr==mask_gt) > threshold_akurasi :
        jumlah_lolos["Otsu"] += 1
    if np.mean(mask_clean_cr==mask_gt) > threshold_akurasi :
        jumlah_lolos["Otsu_Morph"] += 1
    jumlah_lolos["Total"] += 1
    akurasi_otsu.append(np.mean(mask_otsu_cr==mask_gt))
    akurasi_otsu_morph.append(np.mean(mask_clean_cr==mask_gt))
    
    #----------------------------------------------------------------------------------

    for ax in axes.flat: ax.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_P3, "overview", f"{stem}_overview.png"),
                dpi=100, bbox_inches="tight")
    plt.close(fig)   # WAJIB saat loop

# ==========================================
# RINGKASAN RATA-RATA
# ==========================================
keys = ["IoU", "Dice", "Precision", "Recall"]
print("\n=== Rata-rata Metrik Percobaan 3 (Cr YCrCb) ===")
for nama_opsi, metric_list in all_metrics_p3.items():
    print(f"\n[{nama_opsi}]")
    for k in keys:
        vals = [m[k] for m in metric_list if m]
        print(f"  {k:10s}: {np.mean(vals):.4f}")
        
print(f"Akurasi Berdasarkan Jumlah Gambar (threshold = {threshold_akurasi}) :\nOtsu\t\t: {jumlah_lolos['Otsu']/jumlah_lolos['Total']}\nOtsu_Morph\t: {jumlah_lolos['Otsu_Morph']/jumlah_lolos['Total']}")
print(f"Rata2 Akurasi :\nOtsu\t\t: {np.mean(akurasi_otsu)}\nOtsu_Morph\t: {np.mean(akurasi_otsu_morph)}")

gambar["Otsu (YCbCr)"] = jumlah_lolos['Otsu']/jumlah_lolos['Total']
gambar["Otsu_Morph (YCbCr)"] = jumlah_lolos['Otsu_Morph']/jumlah_lolos['Total']

rata_akurasi["Otsu (YCbCr)"] = np.mean(akurasi_otsu)
rata_akurasi["Otsu_Morph (YCbCr)"] = np.mean(akurasi_otsu_morph)



=== Rata-rata Metrik Percobaan 3 (Cr YCrCb) ===

[Opsi1_Otsu]
  IoU       : 0.4193
  Dice      : 0.5626
  Precision : 0.5020
  Recall    : 0.7572

[Opsi2_Morph]
  IoU       : 0.4206
  Dice      : 0.5636
  Precision : 0.5017
  Recall    : 0.7612
Akurasi Berdasarkan Jumlah Gambar (threshold = 0.7) :
Otsu		: 0.68
Otsu_Morph	: 0.68
Rata2 Akurasi :
Otsu		: 0.7204799681214074
Otsu_Morph	: 0.7200210618250651


In [20]:
print("="*85)
print(f"| {f'Akurasi Berdasarkan Jumlah Gambar (threshold = {threshold_akurasi})':^81} |")
print("="*85)
print(f"| {'Otsu':^39} | {'K-Means':^39} |")
print("="*85)
for i,j in zip(gambar.keys(),gambar2.keys()) :
    print(f"| {i:<18} | {gambar[i]:>18} | {j:<18} | {gambar2[j]:>18} |")
print("="*85)

print()

print("="*85)
print(f"| {'Rata Rata Akurasi':^81} |")
print("="*85)
print(f"| {'Otsu':^39} | {'K-Means':^39} |")
print("="*85)
for i,j in zip(rata_akurasi.keys(),rata_akurasi2.keys()) :
    print(f"| {i:<18} | {rata_akurasi[i]:>18} | {j:<18} | {rata_akurasi2[j]:>18} |")
print("="*85)


|                Akurasi Berdasarkan Jumlah Gambar (threshold = 0.7)                |
|                  Otsu                   |                 K-Means                 |
| Otsu (Gray)        |               0.96 | K3 C1              |               0.36 |
| Otsu_Morph (Gray)  |               0.96 | K3 C2              |               0.36 |
| Otsu (YCbCr)       |               0.68 | K4 C1              |                0.4 |
| Otsu_Morph (YCbCr) |               0.68 | K4 C2              |               0.52 |

|                                 Rata Rata Akurasi                                 |
|                  Otsu                   |                 K-Means                 |
| Otsu (Gray)        | 0.8365955530730731 | K3 C1              | 0.6101475987904563 |
| Otsu_Morph (Gray)  | 0.8415782966846703 | K3 C2              | 0.6431873498715822 |
| Otsu (YCbCr)       | 0.7204799681214074 | K4 C1              | 0.6503322224392662 |
| Otsu_Morph (YCbCr) | 0.7200210618250651 | K4 C2    

Jadinya kita pakai percobaan 1, Akurasi Paling tinggi di Percobaan 1

In [22]:
print("="*71)
print(f"| {"Perbandingan Akurasi Setiap Gambar Pada Percobaan 1":^67} |")
print("="*71)
print(f"| {"Nama Gambar":^25} | {"Otsu":^18} | {"Otsu Morph":^18} |")
print("="*71)
for i in range(len(nama_gambar)) :
    print(f"| {nama_gambar[i]:<25} | {akurasi_otsu_p1[i]:<18} | {akurasi_otsu_morph_p1[i]:<18} |")
print("="*71)

|         Perbandingan Akurasi Setiap Gambar Pada Percobaan 1         |
|        Nama Gambar        |        Otsu        |     Otsu Morph     |
| cju0sxqiclckk08551ycbwhno | 0.8533947864010308 | 0.8624145108534047 |
| cju1c0qb4tzi308355wtsnp0y | 0.8123763877934842 | 0.8153764484620518 |
| cju1c6yfz42md08550zgoz3pw | 0.813348352576739  | 0.8168684877499296 |
| cju1cj3f0qi5n0993ut8f49rj | 0.9453442783631463 | 0.950765743690272  |
| cju1f15k3w4ct0835cmde6ypo | 0.8983265363521306 | 0.9119290216182355 |
| cju1fr4etsmrr09933u4t4aql | 0.9622891201131427 | 0.9633532006599993 |
| cju2lberzkdzm09938cl40pog | 0.7503600050087653 | 0.7518344603055347 |
| cju2nbdpmlmcj0993s1cht0dz | 0.84089881106229   | 0.846759498578444  |
| cju2ntxtdzlvu0799xl3j9pan | 0.8860236783851505 | 0.8914847760949939 |
| cju2okvco06xc0799kxe5n1qh | 0.7483208528171087 | 0.7507014159118407 |
| cju2p0eveqtdc0835gpi3p93i | 0.876939613173536  | 0.8892015005359056 |
| cju2qqn5ys4uo0988ewrt2ip2 | 0.6091411251155661 | 0.60945710306